## 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## 학교 가입자 수 & 평균 친구 수 관계
### 공통 전제 검증
본 분석(학교 규모↔친구 수 / 친구 수별 특성 차이) 전에, `friend_count`·`signup_at`을 신뢰할 수 있는지 4가지 전제를 먼저 검증한다.

- 전제① 친구가 같은 학교인가
- 전제② friend_count 분포 — 비교 구간을 어떻게 자를까
- 전제③ friend_count가 실제 친구인가
- 전제④ signup_at이 실제 가입 시각인가


### 1. 같은 학교 여부
- 1% 샘플로 확인
- 75.6%가 같은 학교 친구 = 나머지 25%는 근처 학교로 추정 가능

In [ ]:
%%bigquery df_friend_school_check --project your-gcp-project
-- [Q1 전제 검증] 친구가 같은 학교인가?
-- 결정적 1% 샘플로 유저-친구 엣지를 펼쳐 본인 학교 vs 친구 학교 비교
-- 엣지 : 관계를 하나의 연결선으로 세는 단위. 친구 관계 한 쌍으로 생각하면 됨
WITH ego AS (                                    -- 검증용 유저: 친구 있는 유효 유저 1% 샘플
  SELECT
    u.id                            AS user_id,
    g.school_id                     AS ego_school_id,
    JSON_VALUE_ARRAY(u.friend_id_list) AS friend_ids   -- ARRAY<STRING>
  FROM `your-gcp-project.raw.raw_user` u
  JOIN `your-gcp-project.stage.stg_group` g
    ON u.group_id = g.group_id
  WHERE NOT (u.is_superuser OR u.is_staff)             -- 운영자 제외
    AND u.ban_status = 'N'                             -- 유효 유저만
    AND u.friend_id_list IS NOT NULL
    AND u.friend_id_list != '[]'                       -- 친구 있는 유저만
    AND MOD(ABS(FARM_FINGERPRINT(CAST(u.id AS STRING))), 100) = 0  -- 재현 가능한 1% 샘플
),
edges AS (                                       -- 유저-친구 엣지로 펼치기
  SELECT
    e.ego_school_id,
    SAFE_CAST(fid AS INT64) AS friend_id
  FROM ego e, UNNEST(e.friend_ids) AS fid
),
resolved AS (                                    -- 친구의 학교 붙이기
  SELECT
    ed.ego_school_id,
    fg.school_id AS friend_school_id
  FROM edges ed
  LEFT JOIN `your-gcp-project.raw.raw_user` fu ON ed.friend_id = fu.id
  LEFT JOIN `your-gcp-project.stage.stg_group`     fg ON fu.group_id = fg.group_id
)
SELECT
  COUNT(*)                                           AS total_edges,          -- 전체 친구 엣지 수
  COUNTIF(friend_school_id IS NULL)                  AS friend_unresolved,    -- 친구 학교 못 붙인 엣지(탈퇴/미매칭)
  COUNTIF(friend_school_id = ego_school_id)          AS same_school_edges,    -- 같은 학교 엣지
  ROUND(SAFE_DIVIDE(
    COUNTIF(friend_school_id = ego_school_id),
    COUNTIF(friend_school_id IS NOT NULL)) * 100, 1) AS same_school_pct       -- 해석 가능한 엣지 중 같은 학교 %
FROM resolved;

In [ ]:
df_friend_school_check

### 2. 친구 수 분포 + 요약 통계
- max 값 (친구가 1341명) = avg_report가 300+ 구간에서 0.00이므로 정상 = 이상치 X
- 0명은 0.4%뿐이라 친구 없는 유저는 문제가 없음

In [ ]:
%%bigquery df_friend_dist --project your-gcp-project
-- [friend_count 분포] 요약통계 + 0명 비율
-- 친구 수는 롱테일이라 평균보다 분위수(median/사분위)로 판단
SELECT
  COUNT(*)                                          AS valid_users,        -- 유효 유저 수
  COUNTIF(friend_count = 0)                         AS zero_friend_users,  -- 친구 0명
  ROUND(COUNTIF(friend_count = 0) / COUNT(*) * 100, 1) AS zero_pct,        -- 0명 비율 %
  ROUND(AVG(friend_count), 1)                       AS mean_friends,       -- 평균(참고용)
  APPROX_QUANTILES(friend_count, 100)[OFFSET(25)]   AS p25,               -- 하위 25%
  APPROX_QUANTILES(friend_count, 100)[OFFSET(50)]   AS median,            -- 중앙값
  APPROX_QUANTILES(friend_count, 100)[OFFSET(75)]   AS p75,               -- 상위 25%
  APPROX_QUANTILES(friend_count, 100)[OFFSET(90)]   AS p90,               -- 상위 10%
  APPROX_QUANTILES(friend_count, 100)[OFFSET(99)]   AS p99,               -- 상위 1%
  MAX(friend_count)                                 AS max_friends        -- 최댓값(이상치 확인)
FROM `your-gcp-project.stage.stg_users`
WHERE is_valid_user;

In [ ]:
df_friend_dist

### 분포 시각화

In [ ]:
%%bigquery df_friend_hist_10 --project your-gcp-project
-- [friend_count 히스토그램용] 10명 단위
SELECT
  LEAST(DIV(friend_count, 10) * 10, 150) AS bin_start,   -- 0,10,20,... 150
  COUNT(*)                               AS users
FROM `your-gcp-project.stage.stg_users`
WHERE is_valid_user
GROUP BY bin_start
ORDER BY bin_start;

In [ ]:
h = df_friend_hist_10
labels = [f'{int(b)}+' if b == 150 else f'{int(b)}~{int(b)+9}' for b in h['bin_start']]

plt.figure(figsize=(12, 4.5))
plt.bar(labels, h['users'], color='#4a90d9', width=0.9)

# 기준선: 사분위(p25=31, p75=70)와 중앙값(49)
for x, txt, c in [(31/10, 'p25=31', '#888'), (49/10, 'median=49', '#e74c3c'), (70/10, 'p75=70', '#888')]:
    plt.axvline(x, color=c, linestyle='--', linewidth=1)
    plt.text(x, plt.ylim()[1]*0.9, txt, rotation=90, va='top', fontsize=9, color=c)

plt.xlabel('친구 수 구간'); plt.ylabel('유저 수')
plt.title('friend_count 분포 (10명 단위)')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

### 3. 구간별 유저 수

- 10명 미만 유저를 다 합쳐도 4.3% → 친구 없는 유저 문제는 사실상 없음
- 20+ 구간에 89% 쏠림 → 이 구간으로는 "많은 vs 적은" 비교 불가




In [ ]:
%%bigquery df_friend_buckets --project your-gcp-project
-- [friend_count 구간별 분포] 게이팅(반 4명) 고려해 4에서 한 번 끊음
WITH bucketed AS (
  SELECT
    CASE
      WHEN friend_count = 0            THEN '0'
      WHEN friend_count BETWEEN 1 AND 4  THEN '1-4'
      WHEN friend_count BETWEEN 5 AND 9  THEN '5-9'
      WHEN friend_count BETWEEN 10 AND 19 THEN '10-19'
      WHEN friend_count BETWEEN 20 AND 49 THEN '20-49'
      ELSE '50+'
    END AS friend_bucket
  FROM `your-gcp-project.stage.stg_users`
  WHERE is_valid_user
)
SELECT
  friend_bucket,
  COUNT(*)                                        AS users,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct   -- 전체 대비 비율
FROM bucketed
GROUP BY friend_bucket
ORDER BY
  CASE friend_bucket                              -- 라벨 순서 강제(문자정렬 방지)
    WHEN '0' THEN 0 WHEN '1-4' THEN 1 WHEN '5-9' THEN 2
    WHEN '10-19' THEN 3 WHEN '20-49' THEN 4 ELSE 5
  END;

In [ ]:
df_friend_buckets

In [ ]:
!pip install koreanize_matplotlib

In [ ]:
import koreanize_matplotlib
import matplotlib.pyplot as plt

b = df_friend_buckets
plt.figure(figsize=(9, 4.5))
bars = plt.bar(b['friend_bucket'], b['users'], color='#4a90d9')
for bar, pct in zip(bars, b['pct']):                       # 막대 위 비율 라벨
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{pct}%', ha='center', va='bottom', fontsize=10)
plt.xlabel('친구 수 구간'); plt.ylabel('유저 수')
plt.title('friend_count 구간별 분포')
plt.tight_layout(); plt.show()

### 4. 가입 시기별 친구 수
- 2023-05에 가입 62만 명
- `signup_at`이 실제로 맞는지 확인

In [ ]:
%%bigquery df_friend_by_cohort --project your-gcp-project
-- [friend_count 의미 검증] 가입 시기에 따라 친구가 쌓이는가
-- 오래된 코호트일수록 median↑
SELECT
  DATE_TRUNC(DATE(signup_at), MONTH)              AS signup_month,
  COUNT(*)                                        AS users,
  APPROX_QUANTILES(friend_count, 100)[OFFSET(50)] AS median_friends,
  APPROX_QUANTILES(friend_count, 100)[OFFSET(25)] AS p25,
  APPROX_QUANTILES(friend_count, 100)[OFFSET(75)] AS p75
FROM `your-gcp-project.stage.stg_users`
WHERE is_valid_user
GROUP BY signup_month
ORDER BY signup_month;

In [ ]:
df_friend_by_cohort

In [ ]:
d = df_friend_by_cohort.copy()
d['label'] = d['signup_month'].astype(str).str[:7]         # YYYY-MM

fig, ax1 = plt.subplots(figsize=(12, 5))
# 왼쪽 축: 가입자 수 (로그 스케일 — 628K vs 수백이라 선형이면 나머지가 안 보임)
ax1.bar(d['label'], d['users'], color='#c9d6e5')
ax1.set_yscale('log')
ax1.set_ylabel('가입자 수 (log)', color='#5a6b7d')
ax1.tick_params(axis='x', rotation=45)
# 오른쪽 축: median 친구 수 (선)
ax2 = ax1.twinx()
ax2.plot(d['label'], d['median_friends'], color='#e74c3c', marker='o', linewidth=2)
ax2.set_ylabel('median 친구 수', color='#e74c3c')

plt.title('가입 시기별 가입자 수(막대) & median 친구 수(선)')
fig.tight_layout(); plt.show()

### 5. `signup_at` 의미 확인
- 가입 전 투표 0%, 가입→첫투표 0~2일 → signup_at은 실제 가입 시각 확정
- 2023-05 가입자 628,607명 중 투표 활동이 있는 사람은 13,935명 (약 2.2%)
- 나머지 97.8%는 가입만 하고 투표 기록이 없음

In [ ]:
%%bigquery df_signup_validity --project your-gcp-project
-- [signup_at 진위 검증] 2023-05 대량가입 코호트가 진짜 그때 가입했는가
-- signup 이전에 투표 활동이 있으면 → created_at은 가입시각이 아님(마이그레이션 흔적)
WITH first_activity AS (          -- 유저별 첫 투표(보낸/받은 통합) 시각
  SELECT user_id, MIN(voted_at) AS first_vote_at
  FROM (
    SELECT sender_user_id   AS user_id, voted_at FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id, voted_at FROM `your-gcp-project.stage.stg_vote`
  )
  GROUP BY user_id
)
SELECT
  DATE_TRUNC(DATE(u.signup_at), MONTH)                       AS signup_month,
  COUNT(*)                                                   AS users,
  COUNTIF(fa.first_vote_at < u.signup_at)                    AS vote_before_signup,   -- 가입 전 활동(모순!)
  ROUND(COUNTIF(fa.first_vote_at < u.signup_at)/COUNT(*)*100, 1) AS pct_before,
  APPROX_QUANTILES(                                          -- 가입→첫활동 간격(일) 중앙값
    DATE_DIFF(DATE(fa.first_vote_at), DATE(u.signup_at), DAY), 100)[OFFSET(50)] AS median_gap_days
FROM `your-gcp-project.stage.stg_users` u
JOIN first_activity fa ON u.user_id = fa.user_id
WHERE u.is_valid_user
GROUP BY signup_month
ORDER BY signup_month;

In [ ]:
df_signup_validity

### 가입자 대비 투표자 비율
- 2023년 4~6월 제외, 투표자가 10명 이하
- 실제 수집 기간 확인 결과, 2023년 4월 이전이나 2024년 6월 이후 결과 값 없는 것 맞음

In [ ]:
%%bigquery df_cohort_voting_rate --project your-gcp-project
-- [후기 코호트 검증] 가입자 대비 투표자 비율 (LEFT JOIN이라 미투표 코호트도 안 사라짐)
WITH voters AS (                                  -- 투표 기록 있는 유저(중복 제거)
  SELECT DISTINCT user_id FROM (
    SELECT sender_user_id   AS user_id FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id FROM `your-gcp-project.stage.stg_vote`
  )
)
SELECT
  DATE_TRUNC(DATE(u.signup_at), MONTH)                          AS signup_month,
  COUNT(*)                                                      AS users,          -- 가입자
  COUNTIF(v.user_id IS NOT NULL)                               AS voters,          -- 투표자
  ROUND(COUNTIF(v.user_id IS NOT NULL) / COUNT(*) * 100, 1)     AS voting_rate_pct  -- 투표 전환율
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN voters v ON u.user_id = v.user_id
WHERE u.is_valid_user
GROUP BY signup_month
ORDER BY signup_month;

In [ ]:
df_cohort_voting_rate

In [ ]:
%%bigquery df_vote_range --project your-gcp-project
-- stg_vote 실제 수집 기간 (후기 코호트가 기간 밖인지 확인)
SELECT MIN(voted_at) AS first_vote, MAX(voted_at) AS last_vote
FROM `your-gcp-project.stage.stg_vote`;

In [ ]:
df_vote_range

## Q5. 학교 가입자 수 - 평균 친구 수

- **분석 단위 = 학교** (유저 X)
- 학교 하나를 한 행으로 놓고, "가입자 수 구간별로 평균 친구 수가 어떻게 변하는지 확인
- ※ 가입자 수 기준(활성 유저 아님): 가입만 한 유저도 친구는 맺기 때문

In [ ]:
%%bigquery df_school --project your-gcp-project
-- [Q1] 학교 규모(가입자 수) ↔ 친구 수 관계
-- 학교 단위 집계: 가입자 수 / 평균·중앙값 친구 수 / 침투율(가입자÷재학생)
WITH user_school AS (                              -- 유저 → 학교 매핑 (유효 유저만)
  SELECT
    u.user_id,
    u.friend_count,
    g.school_id
  FROM `your-gcp-project.stage.stg_users` u
  JOIN `your-gcp-project.stage.stg_group` g
    ON u.group_id = g.group_id
  WHERE u.is_valid_user
    AND u.group_id IS NOT NULL                     -- 학급 없는 유저 제외
),
school_agg AS (                                   -- 학교별 집계
  SELECT
    us.school_id,
    COUNT(*)                                          AS signup_users,      -- 가입자 수
    ROUND(AVG(us.friend_count), 1)                    AS avg_friends,       -- 평균 친구 수
    APPROX_QUANTILES(us.friend_count, 100)[OFFSET(50)] AS median_friends,   -- 중앙값(롱테일 대비)
    s.student_count                                   AS student_count      -- 재학생 수(침투율용)
  FROM user_school us
  JOIN `your-gcp-project.stage.stg_school` s
    ON us.school_id = s.school_id
  GROUP BY us.school_id, s.student_count
)
SELECT
  -- 가입자 수 구간: 서비스 게이트(40명) 전후를 세밀하게, 큰 학교는 넓게
  CASE
    WHEN signup_users < 10   THEN '1) <10'
    WHEN signup_users < 40   THEN '2) 10-39'      -- 40명 게이트 아래
    WHEN signup_users < 100  THEN '3) 40-99'      -- 게이트 직후
    WHEN signup_users < 300  THEN '4) 100-299'
    WHEN signup_users < 600  THEN '5) 300-599'
    ELSE                          '6) 600+'
  END                                               AS size_bucket,
  COUNT(*)                                          AS schools,            -- 구간 내 학교 수
  SUM(signup_users)                                 AS total_users,        -- 구간 내 총 가입자
  ROUND(AVG(avg_friends), 1)                        AS avg_friends,        -- 학교평균의 평균
  ROUND(AVG(median_friends), 1)                     AS avg_median_friends, -- 학교중앙값의 평균
  -- 침투율: 학교별 (가입자÷재학생) 평균, student_count 유효한 학교만
  ROUND(AVG(SAFE_DIVIDE(signup_users, NULLIF(student_count, 0))) * 100, 1) AS avg_penetration_pct
FROM school_agg
GROUP BY size_bucket
ORDER BY size_bucket;

- 8.6→16.5→41.9→55.0→57.8%

In [ ]:
df_school

In [ ]:
%%bigquery df_school_check --project your-gcp-project
-- [확인] student_count 신뢰성 + 600+ 학교 존재 여부
WITH user_school AS (
  SELECT u.user_id, g.school_id
  FROM `your-gcp-project.stage.stg_users` u
  JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
  WHERE u.is_valid_user AND u.group_id IS NOT NULL
),
school_agg AS (
  SELECT us.school_id, COUNT(*) AS signup_users, s.student_count
  FROM user_school us
  JOIN `your-gcp-project.stage.stg_school` s ON us.school_id = s.school_id
  GROUP BY us.school_id, s.student_count
)
SELECT
  COUNT(*)                                          AS total_schools,
  COUNTIF(signup_users >= 600)                      AS schools_600plus,       -- 600+ 학교 수
  MAX(signup_users)                                 AS max_signup_users,      -- 최대 가입자
  COUNTIF(student_count IS NULL)                    AS null_student_count,    -- student_count 결측
  COUNTIF(student_count = 0)                        AS zero_student_count,
  COUNTIF(signup_users > student_count)             AS signup_exceeds_enroll, -- 가입자>재학생(모순!)
  ROUND(AVG(student_count), 0)                      AS avg_student_count,
  APPROX_QUANTILES(student_count, 100)[OFFSET(50)]  AS median_student_count
FROM school_agg;

In [ ]:
df_school_check

In [ ]:
d = df_school

fig, ax1 = plt.subplots(figsize=(11, 5.5))

# ── 왼쪽 축: 평균/중앙값 친구 수 (포화 곡선, 주인공) ──
ax1.plot(d['size_bucket'], d['avg_friends'], marker='o', linewidth=2.5,
         color='#2c6fbb', label='평균 친구 수')
ax1.plot(d['size_bucket'], d['avg_median_friends'], marker='s', linewidth=2,
         linestyle='--', color='#6ba3e5', label='중앙값 친구 수')

# 값 라벨 (평균 위에)
for x, y in zip(d['size_bucket'], d['avg_friends']):
    ax1.text(x, y + 1.5, f'{y}', ha='center', fontsize=10, color='#2c6fbb', fontweight='bold')

ax1.set_ylabel('학교 평균 친구 수', color='#2c6fbb')
ax1.set_xlabel('학교 규모 (가입자 수 구간)')
ax1.set_ylim(0, 70)

# ── 포화 시작선 강조 (40-99 → 100-299 사이) ──
ax1.axvspan(1.5, 2.5, color='#ffe8a3', alpha=0.35)          # 급성장 구간 음영

# ── 오른쪽 축: 구간별 학교 수 (맥락, 옅은 막대) ──
ax2 = ax1.twinx()
ax2.bar(d['size_bucket'], d['schools'], alpha=0.15, color='gray', width=0.5)
ax2.set_ylabel('학교 수 (막대, 참고)', color='gray')

ax1.legend(loc='upper left')
plt.title('학교 규모 ↔ 평균 친구 수')
fig.tight_layout()
plt.show()

### 결론

학교 가입자 수가 많을수록 평균 친구 수도 많다. **단 선형이 아니라 포화 곡선**이며,
가입자 ~100명을 넘으면 평균 친구 수가 55명 선에서 완만해짐

| 학교 규모(가입자) | 평균 친구 수 | 해석 |
|---|---|---|
| <10명 | 8.6 | 관계망 형성 실패 |
| 10-39명 | 16.5 | 게이트 아래, 여전히 낮음 |
| 40-99명 | 41.9 | 급성장 구간 |
| 100-299명 | 55.0 | 포화 시작 |
| 300-599명 | 57.8 | 포화(평평) |

- 평균과 중앙값이 거의 나란히 이동 → 소수 인싸가 평균을 끌어올린 것이 아니라 분포 전체가 고르게 상승
- ~친구망 자생 임계(≈100명)는 **서비스 게이트(40명)의 약 2.5배.**~ 40명은 질문이 겨우 실행되는 최소선일 뿐, 관계망이 포화되는 선은 아님
- <10명 학교가 763개 존재 → "친구망을 못 키운 소규모 학교"가 실재하는 타겟군

### 한계

- **관찰 데이터 · 횡단면(스냅샷).** friend_count는 덤프 시점 값 → "규모가 크면 친구가 많다"는 **연관**이며, "가입자가 늘어서 친구가 늘었다"는 시간·인과 단정은 불가.
- **근처 학교 친구 25%.** 친구의 75.6%만 같은 학교(전제① 검증). 나머지는 전화번호 초대 등 학교 밖 경로라, 학교 규모↔친구 수 연결이 100% 순수하진 않음.
- **침투율(가입자÷재학생) 미사용.** accounts_school.student_count가 전교생 수가 아닌 것으로 보임(median 108, 가입자 수와 유사) → "시장 침투율" 해석 불가로 판단해 지표에서 제외

### 의사결정 함의

신규 학교 마케팅의 목표선을 **"40명"이 아니라 "100명 도달"로** 설정해야 친구 관계망이 자생한다.
- 40-99명 구간이 급성장 구간이므로, 이 구간 학교에 유입을 집중하면 포화(100명)로 밀어 올리는 효율이 가장 높음.
- <10명 학교(763개)는 관계망 형성 실패 위험군 → 개별 유입보다 "같은 학교 친구 동반 가입" 유도가 필요

## Q6. 친구 수 많은 유저 vs 적은 유저 특성 차이
- 가입만 하고 미사용한 유저가 다수이므로, 실사용자(PING 활동 유저) 내에서 친구 수에 따른 차이 확인
- 실사용자 정의 : 투표를 보냈거나 받은 기록이 있는 유저

- **대상 = 실사용자(PING 발신/수신 기록 있는 유효 유저, 15,326명)**
- 가입만 하고 미사용한 유저가 다수(90%+)이므로, 이들을 제외해 "활동량 유무" 노이즈를 걷어냄.
- → "친구 수가 이탈을 가르는가"가 아닌, **"활성 유저 중 친구 수가 참여·수익의 강도·질을 가르는가"**

- **비교 구간(실사용자 기준 사분위):** 상위 25% 친구 81명+ vs 하위 25% 친구 40명
- **검증 3단계:** ① 절대수 비교 → ② 받은 PING 대비 비율 → ③ 코호트 통제

### 실사용자 확인

In [ ]:
%%bigquery df_active_friend_dist --project your-gcp-project
-- 실사용자 = PING 보냈거나 받은 기록 있는 유저
WITH active_users AS (                             -- 실사용자 id 집합
  SELECT DISTINCT user_id FROM (
    SELECT sender_user_id   AS user_id FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id FROM `your-gcp-project.stage.stg_vote`
  )
),
active_with_friends AS (                           -- 실사용자 + 친구 수 (유효 유저만)
  SELECT u.friend_count
  FROM `your-gcp-project.stage.stg_users` u
  JOIN active_users a ON u.user_id = a.user_id
  WHERE u.is_valid_user
)
SELECT
  COUNT(*)                                          AS active_users,       -- 실사용자 수
  COUNTIF(friend_count = 0)                         AS zero_friend,        -- 친구 0명
  ROUND(COUNTIF(friend_count = 0)/COUNT(*)*100, 1)  AS zero_pct,
  ROUND(AVG(friend_count), 1)                       AS mean_friends,
  APPROX_QUANTILES(friend_count, 100)[OFFSET(25)]   AS p25,               -- ← Q2 하위 기준
  APPROX_QUANTILES(friend_count, 100)[OFFSET(50)]   AS median,
  APPROX_QUANTILES(friend_count, 100)[OFFSET(75)]   AS p75,               -- ← Q2 상위 기준
  APPROX_QUANTILES(friend_count, 100)[OFFSET(90)]   AS p90,
  MAX(friend_count)                                 AS max_friends
FROM active_with_friends;

In [ ]:
df_active_friend_dist

### 실사용자 내 상위 25% vs 하위 25% 특성 비교

[활동량]
친구 많은 그룹이 높음

- 받은 PING: 113.1 vs 45.5 (2.5배) — 친구 많으면 남이 나를 뽑을 기회 자체가 많다는 것
- 보낸 PING: 86.1 vs 59.7 (1.4배)
- 힌트 열람: 4.2 vs 3.1, 답장: 7.8 vs 6.9

[수익]
친구 많은 그룹이 결제율 1.5배

- 결제율: 9.1% vs 5.9%
- 평균 하트: 112 vs 60 (약 1.9배)
- 결제자 수도 361 vs 235

[부정적 신호]
친구 많은 그룹이 hide가 3배 이상 높음

- avg_hide: 1.55 vs 0.47
- block(0.03 vs 0.02), report(0.06 vs 0.02)는 둘 다 미미한 편

In [ ]:
%%bigquery df_compare --project your-gcp-project
-- 실사용자 내 친구 상위25%(81+) vs 하위25%(40-) 특성 비교
-- 축: 부정신호 / 활동량(PING·힌트·답장) / 수익(결제)
WITH active_users AS (                             -- 실사용자 집합
  SELECT DISTINCT user_id FROM (
    SELECT sender_user_id   AS user_id FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id FROM `your-gcp-project.stage.stg_vote`
  )
),
base AS (                                          -- 실사용자 + 친구그룹 라벨 (중간50% 제외)
  SELECT
    u.user_id,
    u.friend_count,
    u.block_count, u.hide_count, u.report_count,
    CASE
      WHEN u.friend_count >= 81 THEN '상위25%(81+)'
      WHEN u.friend_count <= 40 THEN '하위25%(40-)'
    END AS friend_group
  FROM `your-gcp-project.stage.stg_users` u
  JOIN active_users a ON u.user_id = a.user_id
  WHERE u.is_valid_user
    AND (u.friend_count >= 81 OR u.friend_count <= 40)   -- 상·하위만
),
sent AS (                                          -- 유저별 보낸 PING·답장 수
  SELECT sender_user_id AS user_id,
         COUNT(*) AS sent_pings,
         COUNTIF(is_answered) AS answered_pings
  FROM `your-gcp-project.stage.stg_vote`
  GROUP BY sender_user_id
),
recv AS (                                          -- 유저별 받은 PING·힌트열람 수
  SELECT receiver_user_id AS user_id,
         COUNT(*) AS received_pings,
         COUNTIF(is_hint_opened) AS hint_opened
  FROM `your-gcp-project.stage.stg_vote`
  GROUP BY receiver_user_id
),
pay AS (                                           -- 유저별 결제 건수·총액(하트)
  SELECT user_id,
         COUNT(*) AS pay_count,
         SUM(heart_amount) AS total_heart
  FROM `your-gcp-project.stage.stg_payment`
  GROUP BY user_id
)
SELECT
  b.friend_group,
  COUNT(*)                                              AS users,               -- 그룹 크기
  -- ── 부정신호 (평균) ──
  ROUND(AVG(b.block_count), 2)                          AS avg_block,
  ROUND(AVG(b.hide_count), 2)                           AS avg_hide,
  ROUND(AVG(b.report_count), 2)                         AS avg_report,
  -- ── 활동량 (평균, 기록 없으면 0) ──
  ROUND(AVG(COALESCE(s.sent_pings, 0)), 1)              AS avg_sent,
  ROUND(AVG(COALESCE(r.received_pings, 0)), 1)          AS avg_received,
  ROUND(AVG(COALESCE(r.hint_opened, 0)), 1)             AS avg_hint_opened,
  ROUND(AVG(COALESCE(s.answered_pings, 0)), 1)          AS avg_answered,
  -- ── 수익 ──
  ROUND(COUNTIF(p.user_id IS NOT NULL)/COUNT(*)*100, 1) AS pay_rate_pct,         -- 결제자 비율
  COUNTIF(p.user_id IS NOT NULL)                        AS payers,              -- 결제자 수(표본 확인용)
  ROUND(AVG(COALESCE(p.total_heart, 0)), 0)             AS avg_heart_all        -- 전원 평균(미결제=0)
FROM base b
LEFT JOIN sent s ON b.user_id = s.user_id
LEFT JOIN recv r ON b.user_id = r.user_id
LEFT JOIN pay  p ON b.user_id = p.user_id
GROUP BY b.friend_group
ORDER BY b.friend_group;

In [ ]:
df_compare

### 받은 PING 대비 계산

[힌트 열람]
- 절대적 힌트 열람 비율: 4.2 vs 3.1 (친구 많은 쪽이 높음)
- 비율로 확인: 4.2% vs 8.5% (친구 적은 쪽이 2배 높음)

[숨김]
- 절댓값: 1.55 vs 0.47 (3.3배)
- 비율: 29.92% vs 8.58% (3.5배)

[답장률]
- 9.5% vs 11.3% — 친구 적은 쪽이 조금 높음. 보조 근거 정도...

[결론]
친구 적은 유저 = 받는 게 귀해서 한 건 한 건 소중히 반응 / 친구 많은 유저 = 흔해서 무뎌지고 선별적으로 걸러냄. 참여의 "질"이 다름

In [ ]:
%%bigquery df_rate --project your-gcp-project
WITH active_users AS (
  SELECT DISTINCT user_id FROM (
    SELECT sender_user_id   AS user_id FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id FROM `your-gcp-project.stage.stg_vote`
  )
),
base AS (
  SELECT
    u.user_id, u.friend_count, u.hide_count,
    CASE WHEN u.friend_count >= 81 THEN '상위25%(81+)'
         WHEN u.friend_count <= 40 THEN '하위25%(40-)' END AS friend_group
  FROM `your-gcp-project.stage.stg_users` u
  JOIN active_users a ON u.user_id = a.user_id
  WHERE u.is_valid_user AND (u.friend_count >= 81 OR u.friend_count <= 40)
),
recv AS (                                          -- 받은 PING·힌트열람 (분모/분자)
  SELECT receiver_user_id AS user_id,
         COUNT(*) AS received_pings,
         COUNTIF(is_hint_opened) AS hint_opened
  FROM `your-gcp-project.stage.stg_vote`
  GROUP BY receiver_user_id
),
sent AS (                                          -- 보낸 PING·답장
  SELECT sender_user_id AS user_id,
         COUNT(*) AS sent_pings,
         COUNTIF(is_answered) AS answered_pings
  FROM `your-gcp-project.stage.stg_vote`
  GROUP BY sender_user_id
),
per_user AS (                                      -- 유저 단위로 비율 먼저 계산 (유저별 평균의 평균)
  SELECT
    b.friend_group,
    -- 받은 것 대비 힌트 열람률: 받는 게 흔해지면 하나하나 덜 소중해지나?
    SAFE_DIVIDE(r.hint_opened, r.received_pings)   AS hint_open_rate,
    -- 받은 것 대비 숨김률: 절대 hide차이가 "많이 받아서"인지 분리
    SAFE_DIVIDE(b.hide_count, r.received_pings)     AS hide_rate,
    -- 받은 것 대비 답장률: 받은 칭찬에 반응하는 비율
    SAFE_DIVIDE(r.received_pings - 0, NULLIF(r.received_pings,0)) AS _dummy, -- (자리표시)
    SAFE_DIVIDE(s.answered_pings, s.sent_pings)     AS answer_rate
  FROM base b
  LEFT JOIN recv r ON b.user_id = r.user_id
  LEFT JOIN sent s ON b.user_id = s.user_id
)
SELECT
  friend_group,
  COUNT(*)                                    AS users,
  ROUND(AVG(hint_open_rate) * 100, 1)         AS hint_open_rate_pct,   -- 받은 PING당 힌트 열람 %
  ROUND(AVG(hide_rate) * 100, 2)              AS hide_rate_pct,        -- 받은 PING당 숨김 %
  ROUND(AVG(answer_rate) * 100, 1)            AS answer_rate_pct       -- 보낸 PING당 답장 %
FROM per_user
GROUP BY friend_group
ORDER BY friend_group;

In [ ]:
df_rate

### 같은 가입월 내 비율 격차

[힌트 열람]
- 2023-04: 상위 3.2% vs 하위 8.4% (하위가 2.6배)
- 2023-05: 상위 4.3% vs 하위 8.5% (하위가 2배)

[숨김]
- 2023-05: 상위 33.26% vs 하위 7.17%

In [ ]:
%%bigquery df_cohort --project your-gcp-project
-- [Q2 코호트 통제] 같은 가입월 내에서도 비율 격차가 유지되나?
-- 유지되면 "친구 효과 ≠ 계정 나이 효과" 확정
WITH active_users AS (
  SELECT DISTINCT user_id FROM (
    SELECT sender_user_id   AS user_id FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id FROM `your-gcp-project.stage.stg_vote`
  )
),
base AS (
  SELECT
    u.user_id,
    DATE_TRUNC(DATE(u.signup_at), MONTH) AS signup_month,   -- 코호트 축
    u.hide_count,
    CASE WHEN u.friend_count >= 81 THEN '상위'
         WHEN u.friend_count <= 40 THEN '하위' END AS friend_group
  FROM `your-gcp-project.stage.stg_users` u
  JOIN active_users a ON u.user_id = a.user_id
  WHERE u.is_valid_user AND (u.friend_count >= 81 OR u.friend_count <= 40)
),
recv AS (
  SELECT receiver_user_id AS user_id,
         COUNT(*) AS received_pings,
         COUNTIF(is_hint_opened) AS hint_opened
  FROM `your-gcp-project.stage.stg_vote` GROUP BY receiver_user_id
),
per_user AS (
  SELECT
    b.signup_month, b.friend_group,
    SAFE_DIVIDE(r.hint_opened, r.received_pings) AS hint_open_rate,
    SAFE_DIVIDE(b.hide_count, r.received_pings)   AS hide_rate
  FROM base b
  LEFT JOIN recv r ON b.user_id = r.user_id
)
SELECT
  signup_month,
  friend_group,
  COUNT(*)                              AS users,
  ROUND(AVG(hint_open_rate) * 100, 1)   AS hint_open_rate_pct,
  ROUND(AVG(hide_rate) * 100, 2)        AS hide_rate_pct
FROM per_user
WHERE signup_month IS NOT NULL
GROUP BY signup_month, friend_group
HAVING COUNT(*) >= 30                   -- 그룹당 30명 미만 코호트는 노이즈라 제외
ORDER BY signup_month, friend_group;

In [ ]:
df_cohort

In [ ]:
import numpy as np

c = df_compare.set_index('friend_group')
metrics = ['avg_sent', 'avg_received', 'avg_hint_opened', 'avg_answered']
labels  = ['보낸 PING', '받은 PING', '힌트 열람', '답장']

x = np.arange(len(metrics)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, c.loc['상위25%(81+)', metrics], w, label='상위25% (친구 81+)', color='#2c6fbb')
ax.bar(x + w/2, c.loc['하위25%(40-)', metrics], w, label='하위25% (친구 40-)', color='#f0a04b')

for i, m in enumerate(metrics):                       # 값 라벨
    ax.text(i - w/2, c.loc['상위25%(81+)', m], f"{c.loc['상위25%(81+)', m]}", ha='center', va='bottom', fontsize=9)
    ax.text(i + w/2, c.loc['하위25%(40-)', m], f"{c.loc['하위25%(40-)', m]}", ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('유저당 평균 (건)')
ax.set_title('① 절대수: 친구 많은 유저가 모든 활동에서 높아 보임')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
r = df_rate.set_index('friend_group')
metrics = ['hint_open_rate_pct', 'hide_rate_pct', 'answer_rate_pct']
labels  = ['힌트 열람률', '숨김률', '답장률']

x = np.arange(len(metrics)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, r.loc['상위25%(81+)', metrics], w, label='상위25% (친구 81+)', color='#2c6fbb')
b2 = ax.bar(x + w/2, r.loc['하위25%(40-)', metrics], w, label='하위25% (친구 40-)', color='#f0a04b')

for bars in (b1, b2):                                 # 값 라벨(%)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{bar.get_height()}%',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('받은/보낸 PING 대비 비율 (%)')
ax.set_title('② 비율: 힌트 열람률은 오히려 하위가 높다 (활동량 착시 제거)')
ax.legend()
plt.tight_layout(); plt.show()

### 결론
- 친구가 많을수록 활동 총량은 크지만 참여의 "질"은 낮다

| 지표 | 상위25%(81+) | 하위25%(40-) | 해석 |
|---|---|---|---|
| 받은 PING (절대) | 113.1 | 45.5 | 친구 많을수록 받을 기회 2.5배 |
| 결제율 | 9.1% | 5.9% | 친구 많을수록 결제도 1.5배 |
| **힌트 열람률** (받은 PING 대비) | **4.2%** | **8.5%** | **역전 — 적을수록 정성껏 확인** |
| 숨김률 (받은 PING 대비) | 29.9% | 8.6% | 친구 많을수록 선별적으로 걸러냄 |

**핵심:** 절대수로는 친구 많은 유저가 모든 면에서 활발해 보이지만, 받은 PING 대비 **비율로 보면 힌트 열람률이 역전**됨(4.2% vs 8.5%, 2배)
- 칭찬이 흔해질수록 하나하나에 무뎌짐... → "친구 많은 유저 = 좋은 유저"라는 단순 가정은 성립하지 않음

### 한계

- **관찰 데이터.** 실험이 없어 "친구가 많아서 무뎌진다"는 **인과 단정 불가.** "친구 수와 참여 질이 반대로 움직인다"는 **연관**까지만 서술.
- **실사용자 한정 결과.** 미사용자(90%+) 제외 후의 그림이므로, 전체 유저로 일반화 불가.
- **숨김률 소표본 변동.** 소규모 코호트에서 방향이 뒤집혀, 숨김 관련 결론은 2023-05 기준으로만 신뢰
- **역방향/제3변수 가능성.** "무뎌져서 친구를 방치"인지 "친구가 많아 무뎌짐"인지 관찰로는 분리 불가

### 의사결정 함의

- **친구 수를 "참여 질"의 대리지표로 쓰지 말 것.** 친구 많은 유저는 활동 총량은 크지만 반응 밀도(힌트 열람률)는 오히려 낮으므로, 친구 수만으로 핵심 유저를 판별하면 오판 위험.
- **유저 가치 지표는 "총량"과 "밀도"를 분리해서 볼 것.** 받은 PING 수(총량)와 받은 PING 대비 열람률(밀도)은 친구 수에 따라 반대로 움직임 → 두 축을 함께 봐야 유저를 정확히 이해.
- **추후 실험 과제(본 분석 범위 밖):** 친구 많은 유저의 무뎌짐이 알림 과부하 때문인지 등은 A/B 테스트로 검증 필요. 관찰 데이터로는 제언 불가.

In [ ]:
%%bigquery df_join_dup_check --project your-gcp-project
-- [조인 중복 검증] group 조인이 유저를 뻥튀기하지 않았나?
SELECT
  -- stg_group PK 유일성
  (SELECT COUNT(*) - COUNT(DISTINCT group_id)
   FROM `your-gcp-project.stage.stg_group`)                    AS group_pk_dup,      -- 0이어야
  -- 조인 전 유저 수
  (SELECT COUNT(*) FROM `your-gcp-project.stage.stg_users`
   WHERE is_valid_user AND group_id IS NOT NULL)              AS users_before_join,
  -- 조인 후 유저 수 (늘면 중복)
  (SELECT COUNT(*)
   FROM `your-gcp-project.stage.stg_users` u
   JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
   WHERE u.is_valid_user AND u.group_id IS NOT NULL)          AS users_after_join;   -- 위와 같아야

In [ ]:
df_join_dup_check

In [ ]:
%%bigquery df_friend_outlier --project your-gcp-project
-- [friend_count 상단 이상치] 극단 친구 수 유저가 정상 활동 유저인가?
WITH active_users AS (
  SELECT DISTINCT user_id FROM (
    SELECT sender_user_id   AS user_id FROM `your-gcp-project.stage.stg_vote`
    UNION ALL
    SELECT receiver_user_id AS user_id FROM `your-gcp-project.stage.stg_vote`
  )
)
SELECT
  CASE
    WHEN u.friend_count >= 300 THEN '300+'
    WHEN u.friend_count >= 150 THEN '150-299'
    WHEN u.friend_count >= 81  THEN '81-149 (상위25% 주력)'
  END                                             AS friend_band,
  COUNT(*)                                        AS users,
  ROUND(AVG(u.report_count), 2)                   AS avg_report,      -- 신고 많으면 비정상 의심
  ROUND(AVG(u.block_count), 2)                    AS avg_block,
  ROUND(AVG(u.hide_count), 2)                     AS avg_hide
FROM `your-gcp-project.stage.stg_users` u
JOIN active_users a ON u.user_id = a.user_id
WHERE u.is_valid_user AND u.friend_count >= 81    -- 상위 그룹만
GROUP BY friend_band
ORDER BY friend_band;

In [ ]:
df_friend_outlier

In [ ]:
%%bigquery df_population_def --project your-gcp-project
-- [공통 모수 확정] 각 제외 기준별 유저 수 분해
SELECT
  COUNT(*)                                          AS total_all,          -- 전체
  COUNTIF(is_superuser OR is_staff)                 AS staff_accounts,     -- 운영자
  COUNTIF(ban_status != 'N')                        AS banned,             -- ban
  COUNTIF(NOT (is_superuser OR is_staff))           AS non_staff,          -- 운영자 제외
  COUNTIF(ban_status = 'N')                         AS valid_ban_n,        -- ban 제외
  COUNTIF(ban_status = 'N' AND NOT (is_superuser OR is_staff)) AS valid_and_nonstaff,  -- 둘 다 제외
  COUNT(*) - COUNT(DISTINCT id)                     AS dup_id              -- PK 중복(0이어야)
FROM `your-gcp-project.raw.raw_user`;

In [ ]:
df_population_def